# Clean `single-player-games` for ML (price vs sales proxy)

This notebook turns [`data/single-player-games.csv`](data/single-player-games.csv) into a numeric dataset for modeling.

**Sales proxy:** `steamspy_owners` is parsed from SteamSpy’s **estimated owner range** (e.g. `200,000 .. 500,000`) into `owners_low`, `owners_high`, `owners_mid`, and `log_owners_mid`. These are **not** true unit sales; any model measures association with this proxy, not causal “optimal price.”

**Outputs:** `data/single-player-games-cleaned.parquet` (primary) and optional `data/single-player-games-cleaned.csv`.

## 1. Configuration

Adjust paths and knobs here before running the rest.

In [1]:
from pathlib import Path

SCRIPT_DIR = Path.cwd()
INPUT_CSV = SCRIPT_DIR / "data" / "single-player-games.csv"
OUTPUT_PARQUET = SCRIPT_DIR / "data" / "single-player-games-cleaned.parquet"
OUTPUT_CSV = SCRIPT_DIR / "data" / "single-player-games-cleaned.csv"
WRITE_CSV_MIRROR = True  # set False to skip CSV

# None = today UTC midnight; or set e.g. "2026-05-05" for reproducible age_days
REFERENCE_DATE = None

GENRE_TOP_K = 20
MIN_OWNERS_MID = None  # e.g. 5000.0 to drop very small estimates


## 2. Imports and helpers

The next cell installs `numpy`, `pandas`, and `pyarrow` into the **active kernel** if they are missing (fixes `ModuleNotFoundError` when the kernel is not your project `.venv`).

In [2]:
import importlib.util
import json
import re
import subprocess
import sys
from collections import Counter

_missing = ("numpy", "pandas", "pyarrow", "matplotlib", "seaborn")
if any(importlib.util.find_spec(p) is None for p in _missing):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

OWNERS_PATTERN = re.compile(r"([\d,]+)\s*\.\.\s*([\d,]+)")


def _strip_commas_num(s: str) -> int:
    return int(s.replace(",", "").strip())


def parse_owners_range(raw: object) -> tuple[float | None, float | None]:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return None, None
    text = str(raw).strip()
    if not text:
        return None, None
    m = OWNERS_PATTERN.search(text)
    if not m:
        return None, None
    try:
        low = _strip_commas_num(m.group(1))
        high = _strip_commas_num(m.group(2))
    except ValueError:
        return None, None
    if high < low:
        low, high = high, low
    return float(low), float(high)


def split_semicolon_counts(series: pd.Series) -> pd.Series:
    def count_parts(x: object) -> int:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return 0
        parts = [p.strip() for p in str(x).split(";") if p.strip()]
        return len(parts)

    return series.map(count_parts)


def split_genre_list(raw: object) -> list[str]:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return []
    return [g.strip() for g in str(raw).split(",") if g.strip()]


def sanitize_feature_name(s: str) -> str:
    out = re.sub(r"[^\w]+", "_", s.strip())
    out = re.sub(r"_+", "_", out).strip("_")
    return out or "unknown"


def parse_tags_dict(raw: object) -> dict:
    if raw is None or (isinstance(raw, float) and np.isnan(raw)):
        return {}
    text = str(raw).strip()
    if not text:
        return {}
    try:
        data = json.loads(text)
        return data if isinstance(data, dict) else {}
    except json.JSONDecodeError:
        return {}


## 3. Load raw CSV

In [3]:
df = pd.read_csv(INPUT_CSV, dtype={"appid": "Int64"})
n_read = len(df)
print(f"Loaded {n_read} rows from {INPUT_CSV}")
df.head(2)


Loaded 10739 rows from /Users/mateovargas/Documents/Development/steam-price-analysis/data/single-player-games.csv


,appid,name,release_date,release_date_iso,price_currency,price_final_cents,price_initial_cents,price_discount_percent,price_final_formatted,developers,...,steamspy_average_forever,steamspy_average_2weeks,steamspy_median_forever,steamspy_median_2weeks,steamspy_score_rank,steamspy_price,steamspy_initialprice,steamspy_discount,steamspy_genre,steamspy_tags
0,1313,SiN Gold,"Mar 18, 2020",2020-03-18,USD,999,999,0,$9.99,Ritual Entertainment; Nightdive Studios,...,0.0,0.0,0.0,0.0,NaN,999.0,999.0,0.0,Action,"{""Action"": 167, ""FPS"": 29, ""Cult Classic"": 28,..."
1,7800,Stubbs the Zombie in Rebel Without a Pulse,"Mar 16, 2021",2021-03-16,USD,1999,1999,0,$19.99,Aspyr,...,0.0,0.0,0.0,0.0,NaN,1999.0,1999.0,0.0,Action,"{""Zombies"": 232, ""Funny"": 211, ""Villain Protag..."


## 4. Parse features (owners, dates, price, tags, genres)

In [4]:
# Naive datetimes only (ISO dates have no tz). Avoid tz-aware utcnow() vs naive release_dt.
ref = (
    pd.Timestamp(REFERENCE_DATE).normalize()
    if REFERENCE_DATE
    else pd.Timestamp(pd.Timestamp.now(tz="UTC").date())
)

low_list: list[float | None] = []
high_list: list[float | None] = []
for v in df["steamspy_owners"]:
    lo, hi = parse_owners_range(v)
    low_list.append(lo)
    high_list.append(hi)

df["owners_low"] = low_list
df["owners_high"] = high_list
df["owners_mid"] = (df["owners_low"] + df["owners_high"]) / 2.0
df["log_owners_mid"] = np.where(
    df["owners_mid"].notna() & (df["owners_mid"] > 0),
    np.log(df["owners_mid"]),
    np.nan,
)

df["release_dt"] = pd.to_datetime(df["release_date_iso"], errors="coerce")
df["release_year"] = df["release_dt"].dt.year
df["age_days"] = (ref - df["release_dt"]).dt.days

df["price_usd"] = df["price_final_cents"].astype("float64") / 100.0

df["developer_count"] = split_semicolon_counts(df["developers"])
df["publisher_count"] = split_semicolon_counts(df["publishers"])

tags_parsed = df["steamspy_tags"].map(parse_tags_dict)
df["tag_count"] = tags_parsed.map(len)
df["has_tags"] = df["tag_count"] > 0

df["genre_list"] = df["steamspy_genre"].map(split_genre_list)
df["primary_genre"] = df["genre_list"].map(lambda xs: xs[0] if xs else np.nan)


## 5. Filter rows (sequential drops)

In [5]:
mask_owners = df["owners_low"].notna() & df["owners_high"].notna()
n_bad_owners = int((~mask_owners).sum())
df = df.loc[mask_owners].copy()

mask_price = df["price_final_cents"].notna()
n_bad_price = int((~mask_price).sum())
df = df.loc[mask_price].copy()

mask_date = df["release_dt"].notna()
n_bad_date = int((~mask_date).sum())
df = df.loc[mask_date].copy()

if MIN_OWNERS_MID is not None:
    mask_min = df["owners_mid"] >= MIN_OWNERS_MID
    n_min_owners = int((~mask_min).sum())
    df = df.loc[mask_min].copy()
else:
    n_min_owners = 0

print(f"Reference date (age_days): {ref.date()}")
print(f"Dropped (unparseable owners): {n_bad_owners}")
print(f"Dropped (missing price): {n_bad_price}")
print(f"Dropped (invalid release date): {n_bad_date}")
if MIN_OWNERS_MID is not None:
    print(f"Dropped (owners_mid < {MIN_OWNERS_MID}): {n_min_owners}")
print(f"Remaining rows: {len(df)}")


Reference date (age_days): 2026-05-06
Dropped (unparseable owners): 3
Dropped (missing price): 0
Dropped (invalid release date): 0
Remaining rows: 10736


## 6. Top-K genre multi-hot columns

In [6]:
k = max(0, GENRE_TOP_K)
genre_counter: Counter[str] = Counter()
for lst in df["genre_list"]:
    for g in lst:
        genre_counter[g] += 1
top_genres = [g for g, _ in genre_counter.most_common(k)]

used_names: dict[str, str] = {}
genre_cols: list[str] = []
for g in top_genres:
    base = sanitize_feature_name(g)
    name = base
    i = 2
    while name in used_names.values():
        name = f"{base}_{i}"
        i += 1
    used_names[g] = name
    col = f"genre__{name}"
    genre_cols.append(col)
    df[col] = df["genre_list"].map(lambda lst, gg=g: int(gg in lst))

df_clean = df.drop(columns=["genre_list"])
print(f"Genre multi-hot columns: {len(genre_cols)}")


Genre multi-hot columns: 18


## 7. Validate, save, quick peek

**Column glossary (added / key fields):**

| Column | Meaning |
|--------|---------|
| `owners_mid`, `log_owners_mid` | Midpoint of SteamSpy owner range; log for skewed targets |
| `price_usd`, `price_discount_percent` | Store price and discount |
| `release_dt`, `release_year`, `age_days` | Parsed release date and age vs reference |
| `steamspy_ccu`, `steamspy_median_*` | Engagement (many zeros are normal) |
| `developer_count`, `publisher_count` | Count of `;`-separated names |
| `tag_count`, `has_tags` | Parsed JSON tag dict size |
| `primary_genre` | First genre in SteamSpy genre string |
| `genre__*` | Multi-hot for top-K genres |


In [7]:
required = [
    "appid",
    "owners_mid",
    "log_owners_mid",
    "price_usd",
    "price_discount_percent",
    "release_dt",
    "age_days",
]
missing = [c for c in required if c not in df_clean.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

OUTPUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_parquet(OUTPUT_PARQUET, index=False)
print(f"Wrote Parquet: {OUTPUT_PARQUET} ({len(df_clean)} rows)")

if WRITE_CSV_MIRROR:
    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    df_clean.to_csv(OUTPUT_CSV, index=False)
    print(f"Wrote CSV: {OUTPUT_CSV}")

df_clean[["name", "owners_mid", "price_usd", "age_days", "primary_genre"]].head()


Wrote Parquet: /Users/mateovargas/Documents/Development/steam-price-analysis/data/single-player-games-cleaned.parquet (10736 rows)
Wrote CSV: /Users/mateovargas/Documents/Development/steam-price-analysis/data/single-player-games-cleaned.csv


,name,owners_mid,price_usd,age_days,primary_genre
0,SiN Gold,150000.0,9.99,2240,Action
1,Stubbs the Zombie in Rebel Without a Pulse,350000.0,19.99,1877,Action
2,Second Sight,35000.0,9.99,1854,Action
3,Grand Theft Auto IV: The Complete Edition,7500000.0,19.99,2234,Action
4,Big Mutha Truckers,10000.0,8.99,332,Racing


In [8]:
df_clean[["owners_mid", "price_usd", "age_days", "steamspy_ccu"]].describe()


,owners_mid,price_usd,age_days,steamspy_ccu
count,1.073600e+04,10736.000000,10736.000000,10736.000000
mean,1.429117e+05,9.295305,3033.194113,47.694672
std,8.034093e+05,9.749060,590.036599,718.570696
min,1.000000e+04,0.490000,6.000000,0.000000
25%,1.000000e+04,2.990000,2963.000000,0.000000
50%,1.000000e+04,5.990000,3162.000000,0.000000
75%,3.500000e+04,12.990000,3387.250000,0.000000
max,3.500000e+07,199.990000,3652.000000,32112.000000


## 8. Visualizations (saved to `visualizations/`)

These are common EDA plots you’ll typically generate before training ML models (distribution checks, target skew, price vs sales proxy, correlations, and category effects).

In [ ]:
from pathlib import Path

VIS_DIR = SCRIPT_DIR / "visualizations"
VIS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Saving plots to: {VIS_DIR}")

# If you open this notebook without re-running earlier cells, reload the cleaned dataset.
if "df_clean" not in globals():
    df_clean = pd.read_parquet(OUTPUT_PARQUET)

# Convenience columns
_df = df_clean.copy()
_df["log_price_usd"] = np.where(_df["price_usd"] > 0, np.log(_df["price_usd"]), np.nan)

# Robust y for plotting (avoid inf)
_df["log_owners_mid"] = np.where(_df["owners_mid"] > 0, np.log(_df["owners_mid"]), np.nan)

# Limit extreme owners for clearer scatter (keep full data for modeling)
owners_cap = _df["owners_mid"].quantile(0.995)
_df_scatter = _df[_df["owners_mid"] <= owners_cap].copy()
print(f"Scatter cap owners_mid at p99.5={owners_cap:,.0f}; rows kept={len(_df_scatter)}")


In [ ]:
# 1) Price distribution
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(_df["price_usd"], bins=60, kde=False, ax=ax)
ax.set_title("Price distribution (USD)")
ax.set_xlabel("price_usd")
ax.set_ylabel("count")
fig.tight_layout()
fig.savefig(VIS_DIR / "price_usd_hist.png", dpi=200)
plt.close(fig)

# 2) Owners distribution (log scale)
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(_df["log_owners_mid"].dropna(), bins=60, kde=False, ax=ax)
ax.set_title("SteamSpy owners_mid distribution (log)")
ax.set_xlabel("log(owners_mid)")
ax.set_ylabel("count")
fig.tight_layout()
fig.savefig(VIS_DIR / "owners_mid_log_hist.png", dpi=200)
plt.close(fig)

# 3) Price vs owners (hexbin-like view using seaborn scatter with alpha)
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    _df_scatter["price_usd"],
    _df_scatter["log_owners_mid"],
    s=10,
    alpha=0.15,
    linewidths=0,
)
ax.set_title("Price vs log(owners_mid) (capped at p99.5 owners)")
ax.set_xlabel("price_usd")
ax.set_ylabel("log(owners_mid)")
fig.tight_layout()
fig.savefig(VIS_DIR / "price_vs_log_owners_scatter.png", dpi=200)
plt.close(fig)

# 4) Discount vs owners
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    _df_scatter["price_discount_percent"],
    _df_scatter["log_owners_mid"],
    s=10,
    alpha=0.15,
    linewidths=0,
)
ax.set_title("Discount % vs log(owners_mid) (capped)")
ax.set_xlabel("price_discount_percent")
ax.set_ylabel("log(owners_mid)")
fig.tight_layout()
fig.savefig(VIS_DIR / "discount_vs_log_owners_scatter.png", dpi=200)
plt.close(fig)

print("Saved: price_usd_hist.png, owners_mid_log_hist.png, price_vs_log_owners_scatter.png, discount_vs_log_owners_scatter.png")


In [ ]:
# 5) Boxplot: price by top genres (primary_genre)
_top_genres = (
    _df["primary_genre"].value_counts(dropna=True).head(12).index.tolist()
)
_df_gen = _df[_df["primary_genre"].isin(_top_genres)].copy()

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=_df_gen,
    x="primary_genre",
    y="price_usd",
    ax=ax,
    showfliers=False,
)
ax.set_title("Price distribution by primary_genre (top 12)")
ax.set_xlabel("primary_genre")
ax.set_ylabel("price_usd")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(VIS_DIR / "price_by_primary_genre_box.png", dpi=200)
plt.close(fig)

# 6) Mean log owners by primary genre (top 12)
fig, ax = plt.subplots(figsize=(10, 5))
mean_by_genre = (
    _df_gen.groupby("primary_genre")["log_owners_mid"].mean().sort_values(ascending=False)
)
mean_by_genre.plot(kind="bar", ax=ax)
ax.set_title("Mean log(owners_mid) by primary_genre (top 12)")
ax.set_xlabel("primary_genre")
ax.set_ylabel("mean log(owners_mid)")
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
fig.savefig(VIS_DIR / "mean_log_owners_by_primary_genre_bar.png", dpi=200)
plt.close(fig)

print("Saved: price_by_primary_genre_box.png, mean_log_owners_by_primary_genre_bar.png")


In [ ]:
# 7) Correlation heatmap for numeric features
num_cols = [
    "owners_mid",
    "log_owners_mid",
    "price_usd",
    "price_discount_percent",
    "age_days",
    "steamspy_ccu",
    "steamspy_average_forever",
    "steamspy_average_2weeks",
    "steamspy_median_forever",
    "steamspy_median_2weeks",
    "developer_count",
    "publisher_count",
    "tag_count",
]
num_cols = [c for c in num_cols if c in _df.columns]

corr = _df[num_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr,
    cmap="vlag",
    center=0,
    annot=False,
    square=False,
    ax=ax,
)
ax.set_title("Correlation heatmap (numeric features)")
fig.tight_layout()
fig.savefig(VIS_DIR / "correlation_heatmap_numeric.png", dpi=200)
plt.close(fig)

print("Saved: correlation_heatmap_numeric.png")
